# SAE-Gated RAG: preliminary probe results

Can a probe on Gemma 3 4B's activation at the end of the question predict whether the model knows the answer, before it generates anything? SAE probe vs dense probe vs question-text baseline.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **on**. No login or upload needed: questions, model and SAEs all come from Hugging Face. Then Run All. Everything for the report lands in `/kaggle/working/results/`.

Label here is the proxy: **ANSWER** = closed-book answer correct, else **NOT_ANSWER**. The RETRIEVE vs ABSTAIN split needs the BM25 index, which is not part of this notebook.

In [ ]:
!pip install -q "transformers>=4.56" accelerate datasets safetensors

In [ ]:
import gc, json, os, random, re, string, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MaxAbsScaler, StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

SMOKE = os.environ.get("SAERAG_SMOKE") == "1"     # small 1B run on CPU, to test the notebook itself
SEED = 13
MODEL = "unsloth/gemma-3-1b-it" if SMOKE else "unsloth/gemma-3-4b-it"   # ungated mirror, same weight hashes as google/
SAE_REPO = "google/gemma-scope-2-1b-it" if SMOKE else "google/gemma-scope-2-4b-it"
LAYERS = [13, 17] if SMOKE else [9, 17, 22, 29]
SAE_ID = "resid_post/layer_{}_width_16k_l0_medium"
N = dict(nq_train=30, tqa_train=30, nq_val=10, tqa_val=10, nq_test=15, tqa_test=15, popqa=15) if SMOKE else \
    dict(nq_train=1000, tqa_train=1000, nq_val=200, tqa_val=200, nq_test=300, tqa_test=300, popqa=300)
BATCH = 4 if SMOKE else 16
MAX_NEW_TOKENS = 32
BUDGETS = [20, 40] if SMOKE else [50, 100, 200, 500, 1000]
SEEDS = [0, 1] if SMOKE else [0, 1, 2, 3, 4]
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
OUT = Path("/kaggle/working/results" if Path("/kaggle/working").exists()
           else os.environ.get("SAERAG_OUT", "results"))
OUT.mkdir(parents=True, exist_ok=True)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not SMOKE:
    assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: 4B in fp32 is about 17 GB and does not fit on one T4"

## Questions
Same sources and scoring as the project (`data/build.py`, `answers.py`, `prompts.py`). This is a fresh sample, not the project's canonical splits, so keep these numbers separate from later runs.

In [ ]:
def aliases(v):
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except json.JSONDecodeError:
            v = [v]
    if isinstance(v, dict):
        v = v.get("aliases") or v.get("value") or []
    return [v] if isinstance(v, str) else list(v)

def sample(ds, n, seed):
    ids = list(range(len(ds)))
    random.Random(seed).shuffle(ids)
    seen, out = set(), []
    for i in ids:
        if ds[i]["question"] not in seen:
            seen.add(ds[i]["question"])
            out.append(ds[i])
        if len(out) == n:
            return out
    raise ValueError("not enough rows")

def rows(ds, name, role, answer_key="answer"):
    return [dict(dataset=name, role=role, question=r["question"], answers=aliases(r[answer_key])) for r in ds]

nq, nq_te = [load_dataset("google-research-datasets/nq_open", split=s) for s in ("train", "validation")]
tq, tq_te = [load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split=s) for s in ("train", "validation")]
pop = load_dataset("akariasai/PopQA", split="test").filter(lambda r: r["s_pop"] is not None and float(r["s_pop"]) < 100)

a = sample(nq, N["nq_train"] + N["nq_val"], SEED)        # train and val are disjoint samples of the source train split
b = sample(tq, N["tqa_train"] + N["tqa_val"], SEED)
Q = (rows(a[:N["nq_train"]], "nq", "train") + rows(b[:N["tqa_train"]], "triviaqa", "train")
     + rows(a[N["nq_train"]:], "nq", "val") + rows(b[N["tqa_train"]:], "triviaqa", "val")
     + rows(sample(nq_te, N["nq_test"], SEED + 2), "nq", "test")
     + rows(sample(tq_te, N["tqa_test"], SEED + 2), "triviaqa", "test")
     + rows(sample(pop, N["popqa"], SEED), "popqa", "popqa", "possible_answers"))
df = pd.DataFrame(Q)
assert df.answers.map(len).gt(0).all()
print(df.groupby(["role", "dataset"]).size())

## Generate closed-book answers and capture activations
One forward pass per question reads the residual stream at the last token of the prompt (decision D5). `hidden_states[L + 1]` is the output of block L, which is the SAEs' hook point. Results are cached, so a rerun of this cell in the same session is instant.

In [ ]:
PUNCT = set(string.punctuation)
def normalize(s):
    s = "".join(c for c in s.lower() if c not in PUNCT)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def is_correct(output, answers):
    out = normalize(output).split()
    for a in answers:
        n = normalize(a).split()
        if n and any(out[i:i + len(n)] == n for i in range(len(out) - len(n) + 1)):
            return True
    return False

CLOSED_BOOK = "Answer the question. Reply with only the answer, in a few words.\n\nQuestion: {question}"
cache = OUT / "activations.npz"
if cache.exists():
    z = np.load(cache, allow_pickle=True)
    assert list(z["questions"]) == df.question.tolist(), "cache is from another sample; delete it"
    acts = {L: z[f"L{L}"] for L in LAYERS}
    df["output"] = z["outputs"]
    timing = json.loads(str(z["timing"]))
else:
    tok = AutoTokenizer.from_pretrained(MODEL)
    tok.padding_side = "left"
    # Gemma 3 overflows in fp16; T4s have no bf16, so fp32 across both cards.
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
    dtype = torch.bfloat16 if bf16 else torch.float32
    kw = dict(device_map="auto", max_memory={i: "12GiB" for i in range(torch.cuda.device_count())}) if torch.cuda.is_available() else {}
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype, **kw).eval()
    n_layers = model.config.get_text_config().num_hidden_layers
    print(MODEL, dtype, n_layers, "layers")

    prompts = [tok.apply_chat_template([{"role": "user", "content": CLOSED_BOOK.format(question=q)}],
                                       tokenize=False, add_generation_prompt=True) for q in df.question]
    order = np.argsort([len(p) for p in prompts])    # similar lengths together, less padding
    store = {L: np.zeros((len(df), model.config.get_text_config().hidden_size), np.float32) for L in LAYERS}
    outputs = [""] * len(df)
    t_fwd = t_gen = 0.0
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        enc = tok([prompts[j] for j in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            hs = model(**enc, output_hidden_states=True).hidden_states
        assert len(hs) == n_layers + 1
        for L in LAYERS:
            store[L][idx] = hs[L + 1][:, -1].float().cpu().numpy()   # left padding: -1 is the last token
        del hs
        t1 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        t2 = time.perf_counter()
        t_fwd += t1 - t0; t_gen += t2 - t1
        for j, t in zip(idx, tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            outputs[j] = t.strip().split("\n")[0].strip()
        if i % (BATCH * 25) == 0:
            print(f"{i}/{len(df)}  forward {t_fwd:.0f}s  generate {t_gen:.0f}s", flush=True)
    acts, df["output"] = store, outputs
    timing = dict(forward_ms_per_q=1000 * t_fwd / len(df), generate_ms_per_q=1000 * t_gen / len(df))
    np.savez(cache, questions=df.question.values, outputs=df.output.values, timing=json.dumps(timing),
             **{f"L{L}": v for L, v in acts.items()})
    del model
    gc.collect(); torch.cuda.empty_cache()

df["correct"] = [is_correct(o, a) for o, a in zip(df.output, df.answers)]
y = df.correct.astype(int).values
print(timing)
accuracy = df.groupby(["role", "dataset"]).correct.agg(n="size", closed_book_accuracy="mean").round(3).reset_index()
print(accuracy.to_string(index=False))
print(df[["dataset", "question", "output", "correct"]].sample(8, random_state=SEED).to_string())

## SAE codes
Pretrained Gemma Scope 2 JumpReLU SAEs, not retrained. **Check:** mean L0 should be near the config's target, and `err_vs_raw` (reconstruction error over the raw activation's energy) well below 1. If not, the hook layer is wrong and nothing below can be trusted. `fvu` subtracts the mean activation: every question is read at the same prompt position, so little variance is left around that mean and `fvu` can sit near 1 even when the SAE is fine.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
codes, sae_rows = {}, []
for L in LAYERS:
    sid = SAE_ID.format(L)
    cfg = json.load(open(hf_hub_download(SAE_REPO, f"{sid}/config.json")))
    p = {k: v.float().to(device) for k, v in load_file(hf_hub_download(SAE_REPO, f"{sid}/params.safetensors")).items()}
    X = torch.from_numpy(acts[L])
    parts, sq_err = [], 0.0
    with torch.no_grad():
        for i in range(0, len(X), 256):
            x = X[i:i + 256].to(device)
            pre = x @ p["w_enc"] + p["b_enc"]
            c = pre * (pre > p["threshold"])
            sq_err += ((x - (c @ p["w_dec"] + p["b_dec"])) ** 2).sum().item()
            parts.append(sp.csr_matrix(c.cpu().numpy()))
    codes[L] = sp.vstack(parts).tocsr()
    fvu = sq_err / ((X - X.mean(0)) ** 2).sum().item()
    err_vs_raw = sq_err / (X ** 2).sum().item()
    sae_rows.append(dict(layer=L, width=codes[L].shape[1], target_l0=cfg["l0"],
                         mean_l0=round(codes[L].getnnz(1).mean(), 1), fvu=round(fvu, 3), err_vs_raw=round(err_vs_raw, 3)))
    del p
    gc.collect(); torch.cuda.empty_cache()
sae_check = pd.DataFrame(sae_rows)
print(sae_check.to_string(index=False))

## Probes
Logistic regression, class-weighted, C chosen by 3-fold CV on macro-F1, same as `probes/train.py`. The layer is chosen on **val**, then the test set is scored once.

In [ ]:
role = df.role.values
idx = {r: np.where(role == r)[0] for r in ("train", "val", "test", "popqa")}
texts = np.array(df.question.tolist(), dtype=object)

def make_model(kind):
    if kind == "majority":
        return DummyClassifier(strategy="most_frequent")
    lr = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=0)
    steps = {"text": [TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), lr],
             "sae": [MaxAbsScaler(), lr], "dense": [StandardScaler(), lr]}[kind]
    return GridSearchCV(make_pipeline(*steps), {"logisticregression__C": C_GRID}, cv=3, scoring="f1_macro", n_jobs=-1)

def feats(kind, L=None):
    return {"sae": codes.get(L), "dense": acts.get(L), "text": texts, "majority": np.zeros((len(y), 1))}[kind]

def metrics(p, t):
    return dict(macro_f1=f1_score(t, p > 0.5, average="macro"), accuracy=accuracy_score(t, p > 0.5),
                auroc=roc_auc_score(t, p) if len(set(t)) == 2 else float("nan"))

def boot_ci(p, t, stat, n=1000):
    rng = np.random.default_rng(SEED)
    out = []
    for _ in range(n):
        s = rng.integers(0, len(t), len(t))
        if len(set(t[s])) == 2:
            out.append(stat(p[s], t[s]))
    return np.percentile(out, [2.5, 97.5], axis=0)

fits, sel_rows = {}, []
for kind in ("sae", "dense"):
    for L in LAYERS:
        X = feats(kind, L)
        m = make_model(kind).fit(X[idx["train"]], y[idx["train"]])
        fits[kind, L] = m
        sel_rows.append(dict(kind=kind, layer=L, **metrics(m.predict_proba(X[idx["val"]])[:, 1], y[idx["val"]])))
selection = pd.DataFrame(sel_rows).round(3)
print(selection.to_string(index=False))
best = {k: int(selection[selection.kind == k].sort_values("macro_f1").layer.iloc[-1]) for k in ("sae", "dense")}
print("chosen on val:", best)

P = {"majority": make_model("majority").fit(feats("majority")[idx["train"]], y[idx["train"]]).predict_proba(feats("majority"))[:, 1],
     "question text (TF-IDF)": make_model("text").fit(texts[idx["train"]], y[idx["train"]]).predict_proba(texts)[:, 1]}
P[f"dense L{best['dense']}"] = fits["dense", best["dense"]].predict_proba(feats("dense", best["dense"]))[:, 1]
P[f"SAE L{best['sae']}"] = fits["sae", best["sae"]].predict_proba(feats("sae", best["sae"]))[:, 1]

res = []
for evalset in ("test", "popqa"):
    te = idx[evalset]
    for name, p in P.items():
        lo, hi = boot_ci(p[te], y[te], lambda a, b: f1_score(b, a > 0.5, average="macro"))
        res.append(dict(eval=evalset, model=name, **metrics(p[te], y[te]), f1_lo=lo, f1_hi=hi))
results = pd.DataFrame(res).round(3)
print(results.to_string(index=False))

sae_p, dense_p = P[f"SAE L{best['sae']}"], P[f"dense L{best['dense']}"]
te = idx["test"]
rng = np.random.default_rng(SEED)
diffs = []
for _ in range(1000):
    s = te[rng.integers(0, len(te), len(te))]
    diffs.append(f1_score(y[s], sae_p[s] > 0.5, average="macro") - f1_score(y[s], dense_p[s] > 0.5, average="macro"))
sae_vs_dense = dict(diff=float(np.mean(diffs)), lo=float(np.percentile(diffs, 2.5)), hi=float(np.percentile(diffs, 97.5)))
print("SAE minus dense macro-F1 on test (paired bootstrap):", sae_vs_dense)

## Label efficiency
The proposal's H2 asks whether SAE beats dense with few labels. Each point is the mean over seeded training subsamples.

In [ ]:
curve = []
for kind, L, name in [("sae", best["sae"], "SAE"), ("dense", best["dense"], "Dense"), ("text", None, "Question text")]:
    X = feats(kind, L)
    for budget in BUDGETS:
        for seed in SEEDS:
            tr, _ = train_test_split(idx["train"], train_size=budget, stratify=y[idx["train"]], random_state=seed)
            p = make_model(kind).fit(X[tr], y[tr]).predict_proba(X[idx["test"]])[:, 1]
            curve.append(dict(model=name, budget=budget, seed=seed, **metrics(p, y[idx["test"]])))
curve = pd.DataFrame(curve)
summary = curve.groupby(["model", "budget"]).macro_f1.agg(["mean", "std"]).round(3).reset_index()
print(summary.pivot(index="budget", columns="model", values="mean").to_string())

fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=200)
for (name, color, marker) in [("SAE", "#2a78d6", "o"), ("Dense", "#eb6834", "s"), ("Question text", "#1baf7a", "^")]:
    s = summary[summary.model == name]
    ax.errorbar(s.budget, s["mean"], yerr=s["std"].fillna(0), color=color, marker=marker, markersize=6, lw=2, capsize=2, label=name)
ax.set_xscale("log"); ax.set_xticks(BUDGETS); ax.set_xticklabels(BUDGETS); ax.minorticks_off()
ax.set_xlabel("Training labels"); ax.set_ylabel("Macro-F1 on held-out test")
ax.set_title("Predicting closed-book correctness from the question", loc="left", fontsize=11)
ax.grid(axis="y", color="#e5e4e0", lw=0.8); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / "label_efficiency.png"); plt.show()

## What the SAE probe looks at
Features with the largest weights, with the questions that activate them most. Look the IDs up on Neuronpedia before claiming what any feature means.

In [ ]:
L = best["sae"]
m = fits["sae", L]
coef = m.best_estimator_[-1].coef_[0]
tr = idx["train"]
top = []
for label, ids in (("ANSWER", np.argsort(-coef)[:10]), ("NOT_ANSWER", np.argsort(coef)[:10])):
    for f in ids:
        col = codes[L][tr, int(f)].toarray().ravel()
        qs = texts[tr][np.argsort(-col)[:3]][col[np.argsort(-col)[:3]] > 0]
        top.append(dict(predicts=label, feature=int(f), weight=round(float(coef[f]), 3),
                        active_on=int((col > 0).sum()), top_questions=" | ".join(qs)))
top = pd.DataFrame(top)
print("non-zero weights:", int((coef != 0).sum()), "of", len(coef))
print(top.to_string(index=False, max_colwidth=90))

## Save for the report

In [ ]:
def table(d):
    d = d.astype(str)
    return "\n".join(["| " + " | ".join(d.columns) + " |", "|" + "---|" * len(d.columns)]
                     + ["| " + " | ".join(r) + " |" for r in d.values]) + "\n"

n = df.groupby("role").size().to_dict()
md = f"""# Preliminary results

**Setup.** {MODEL} (frozen), greedy closed-book answers, activation at the last prompt token. Gemma Scope 2 SAEs ({SAE_REPO}, 16k width, medium L0, layers {LAYERS}), not retrained. Label: closed-book answer correct (ANSWER) or not (NOT_ANSWER), by token-span alias match. Logistic-regression probes on {n['train']} NQ + TriviaQA training questions; layer chosen on {n['val']} validation questions; test on {n['test']} NQ + TriviaQA and {n['popqa']} long-tail PopQA questions. Brackets are 95% bootstrap intervals.

**Limits.** Two-way proxy label only: RETRIEVE vs ABSTAIN and all routing metrics (retrieval calls, hallucination, latency end to end) are not measured here. Questions are a fresh sample, not the project's canonical splits.

## Closed-book accuracy
{table(accuracy)}
## SAE check
{table(sae_check)}
## Probes
{table(results)}
SAE minus dense macro-F1 on test: {sae_vs_dense['diff']:.3f} [{sae_vs_dense['lo']:.3f}, {sae_vs_dense['hi']:.3f}]

Probe forward pass: {timing['forward_ms_per_q']:.0f} ms per question at batch {BATCH}; closed-book generation: {timing['generate_ms_per_q']:.0f} ms per question.

## Label efficiency (mean macro-F1 on test)
{table(summary.pivot(index="budget", columns="model", values="mean").reset_index())}
See label_efficiency.png.

## Top SAE features
{table(top.drop(columns="top_questions"))}"""
(OUT / "results.md").write_text(md)
for name, d in [("accuracy", accuracy), ("sae_check", sae_check), ("layer_selection", selection), ("probes", results),
                ("label_efficiency", curve), ("top_features", top)]:
    d.to_csv(OUT / f"{name}.csv", index=False)
df.drop(columns="answers").to_csv(OUT / "generations.csv", index=False)
print(md)